# boolean-mask-identity-replace — ex9: non-max suppression via iterative mask update

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `boolean-mask-identity-replace`. Running the final beacon cell reports progress against the `Numpy: Indexing and selection` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Indexing and selection` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`boolean-mask-identity-replace`** (exercise 9). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "boolean-mask-identity-replace"
DD_SUBTOPIC = "Numpy: Indexing and selection"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Mask & substitute — quick refresher

**Build a mask.** Any comparison returns a `dtype=bool` tensor of the same shape: `x < 0`, `x.abs() < eps`, `(x > 0) & (x < 1)`. Combine with `&`, `|`, `~`.

**Write through a mask.** `y[mask] = value` modifies in place. Scalars broadcast; tensor values must match the shape of `y[mask]` after broadcasting. Always `clone()` first if the function must not mutate its input.

**The dangerous case.** When a mask is the *wrong* shape, indexing can silently collapse axes or pick the wrong cells. Always check `mask.sum()` and `mask.shape` before trusting the result.

### Exercise 9 — non-max suppression via iterative mask update

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Bloom level: Analyze
> LO: Implement a baseline non-max suppression by iteratively selecting the highest-scoring survivor and masking out its neighbours.
> Keywords: nms, iterative-mask, suppression, object-detection, integrative
> ```

**KCs targeted:** `mask-update-in-a-loop`, `argmax-of-masked-vector`, `mask-from-row-of-matrix`

Implement `ex9_nms(scores, overlap, iou_threshold)`, a baseline NMS that takes:
- `scores`: `(N,)` confidence scores
- `overlap`: `(N, N)` symmetric matrix where `overlap[i, j]` is the IoU between box `i` and box `j`. Diagonal is 1.
- `iou_threshold`: float

Return an `(N,)` boolean tensor `kept` where `kept[i] = True` iff box `i` survives suppression.

**Algorithm** (you must use boolean-mask updates — no `argsort`, no `for box in sorted_boxes` Python list):

```
alive = torch.ones(N, dtype=torch.bool)
kept  = torch.zeros(N, dtype=torch.bool)
while alive.any():
    # 1. pick the alive box with highest score
    masked_scores = scores.clone()
    masked_scores[~alive] = -float('inf')
    i = int(masked_scores.argmax())
    kept[i]  = True
    alive[i] = False
    # 2. suppress every alive box that overlaps too much with i
    suppress  = (overlap[i] > iou_threshold) & alive
    alive[suppress] = False
return kept
```

Implement that. The integrative test will give you a small set of partially-overlapping boxes with hand-checkable survivors.

In [ ]:
def ex9_nms(scores: Tensor, overlap: Tensor, iou_threshold: float) -> Tensor:
    N = scores.shape[0]
    alive = t.ones(N, dtype=t.bool)
    kept = t.zeros(N, dtype=t.bool)
    while alive.any().item():
        masked_scores = scores.clone()
        masked_scores[~alive] = float('-inf')
        i = int(masked_scores.argmax())
        kept[i] = True
        alive[i] = False
        suppress = (overlap[i] > iou_threshold) & alive
        alive[suppress] = False
    return kept


<details><summary>Solution</summary>

```python
def ex9_nms(scores: Tensor, overlap: Tensor, iou_threshold: float) -> Tensor:
    N = scores.shape[0]
    alive = t.ones(N, dtype=t.bool)
    kept = t.zeros(N, dtype=t.bool)
    while alive.any().item():
        masked_scores = scores.clone()
        masked_scores[~alive] = float('-inf')
        i = int(masked_scores.argmax())
        kept[i] = True
        alive[i] = False
        suppress = (overlap[i] > iou_threshold) & alive
        alive[suppress] = False
    return kept
```

**Three boolean-mask moves combined.**
1. *Mask-then-argmax* — `scores[~alive] = -inf; argmax(scores)` picks the highest survivor in one shot.
2. *Mask-from-a-row* — `(overlap[i] > thresh)` gives an `(N,)` bool flagging everything that overlaps box `i`. Combine with `& alive` so we don't 're-suppress' already-dead boxes (idempotent, but tidy).
3. *Mask write-back* — `alive[suppress] = False` flips an arbitrary subset in one indexed assign.

**Why this isn't the production NMS.** Real implementations sort once up front and iterate the sorted list, which avoids re-scanning `scores` each loop. They also operate on (x1, y1, x2, y2) boxes and compute IoU on the fly. The all-boolean version here is pedagogically clean: one mask per logical role.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex9'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex9',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()